# Save pre-processed data per exercise

In [4]:
import os
import sys

sys.path.insert(0, '/home/marcos/Documentos/GitHub/TFM_code/src')

from preprocessing import audio_processor
from data import humv_loader
import numpy as np
import pandas as pd

# Define root directory and output folder
root_directory = '/home/marcos/Documentos/GitHub/TFM_code/data/raw'

output_extension = '5s_with_no_overlap_22kHz_top_db_20'
output_folder = f'/home/marcos/Documentos/GitHub/TFM_code/data/processed/{output_extension}_all_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
audio_length = 5  # seconds
audio_total_duration = 150  # seconds
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    (f"1 audios of {audio_length} seconds", ['aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav', 'ta.wav', 'uy.wav', 'vocal.wav'], 0, audio_length, audio_total_duration),
    (f"3 audios of {audio_length} seconds", ['habla libre.wav', 'robo galletas.wav'], 0, audio_length, audio_total_duration),
    (f"6 audios of {audio_length} seconds", ['fluencia categorial.wav'], 0, audio_length, audio_total_duration),
    (f"10 audios of {audio_length} seconds", ['lectura texto.wav'], 0, audio_length, audio_total_duration),
]

for desc, exact_names_values, start_time, chunk_duration, max_duration in exercise_sets:
    
    print(f"\nProcessing set: {desc}")
    
    for exact_names in exact_names_values:
        print(f"  \n Processing audio files: {exact_names}")
        df = humv_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)
        # df = df[df['Label'].isin([0, 2])]
        # df['Label'] = df['Label'].replace(2, 1)
        
        # List of patient IDs to remove
        patients_to_remove_TFM = ["HUMV_NFC_11","HUMV_HC_25","HUMV_HC_9","HUMV_NFC_1","HUMV_NFC_17","HUMV_HC_15","HUMV_NFC_10","HUMV_NFC_23","HUMV_NFC_5","HUMV_NFC_24","HUMV_HC_4", "HUMV_NFC_21", "HUMV_NFC_16", "HUMV_HC_18", "HUMV_PD_1", "HUMV_PD_2", "HUMV_PD_9", "HUMV_PD_13", "HUMV_AC_2", "HUMV_AC_5", "HUMV_AC_32", "HUMV_AC_24", "HUMV_AC_14"] 

        print("Removing patients:", patients_to_remove_TFM)
        
        # Remove duplicates just in case
        patients_to_remove = list(set(patients_to_remove_TFM))
        
        # Filter the dataframe
        df_filtered = df[~df['Patient'].isin(patients_to_remove_TFM)].copy()

        value_counts = df_filtered['Label'].value_counts()
        print(value_counts) 

        audio_segments, labels_np, patient_ids, exercises = audio_processor.execute_preprocess_and_split(
            df_filtered,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=None, # max_duration, None
            target_sr=22050,
            remove_silence=True,
            top_db=20,
            silence_duration=2,
            file_path_column='File_Path',
            label_column='Label',
            patient_column='Patient',
            audio_name_column='Audio_Name',
            overlap=0,
            min_chunk_ratio=0.75,
        )
        file_name_without_extension = os.path.splitext(exact_names)[0]
        np.save(os.path.join(output_folder, f"audio_segments_{output_extension}_{file_name_without_extension}.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_{output_extension}_{file_name_without_extension}.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_{output_extension}_{file_name_without_extension}.npy"), patient_ids)
        np.save(os.path.join(output_folder, f"exercises_{output_extension}_{file_name_without_extension}.npy"), exercises)
        print(f"Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")


Processing set: 1 audios of 5 seconds
  
 Processing audio files: aueoi.wav
Loaded 114 audio files.
Label distribution:
Label
0    49
1    34
2    31
Name: count, dtype: int64
Removing patients: ['HUMV_NFC_11', 'HUMV_HC_25', 'HUMV_HC_9', 'HUMV_NFC_1', 'HUMV_NFC_17', 'HUMV_HC_15', 'HUMV_NFC_10', 'HUMV_NFC_23', 'HUMV_NFC_5', 'HUMV_NFC_24', 'HUMV_HC_4', 'HUMV_NFC_21', 'HUMV_NFC_16', 'HUMV_HC_18', 'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13', 'HUMV_AC_2', 'HUMV_AC_5', 'HUMV_AC_32', 'HUMV_AC_24', 'HUMV_AC_14']
Label
0    35
1    29
2    29
Name: count, dtype: int64
Total chunks generated: 206
Unique exercises: ['aueoi']
Arrays saved successfully in /home/marcos/Documentos/GitHub/TFM_code/data/processed/5s_with_no_overlap_22kHz_top_db_20_all_audio!
  
 Processing audio files: ka.wav
Loaded 118 audio files.
Label distribution:
Label
0    50
1    34
2    34
Name: count, dtype: int64
Removing patients: ['HUMV_NFC_11', 'HUMV_HC_25', 'HUMV_HC_9', 'HUMV_NFC_1', 'HUMV_NFC_17', 'HUMV_HC_15',

# AC vs PD

In [7]:
import os
import sys

sys.path.insert(0, '/home/marcos/Documentos/GitHub/TFM_code/src')

from preprocessing import audio_processor
from data import humv_loader
import numpy as np
import pandas as pd

# Define root directory and output folder
root_directory = '/home/marcos/Documentos/GitHub/TFM_code/data/raw'

output_extension = '5s_with_no_overlap_16kHz_top_db_20_mean_audio'
output_folder = f'/home/marcos/Documentos/GitHub/TFM_code/data/processed/AC_vs_PD/{output_extension}'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
audio_length = 5  # seconds
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    (f"1 audios of {audio_length} seconds", ['aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav', 'ta.wav', 'uy.wav', 'vocal.wav'], 0, audio_length, 10),
    (f"3 audios of {audio_length} seconds", ['habla libre.wav', 'robo galletas.wav'], 0, audio_length, 35),
    (f"6 audios of {audio_length} seconds", ['fluencia categorial.wav'], 0, audio_length, 60),
    (f"10 audios of {audio_length} seconds", ['lectura texto.wav'], 0, audio_length, 150),
]

for desc, exact_names_values, start_time, chunk_duration, max_duration in exercise_sets:
    
    print(f"\nProcessing set: {desc}")
    
    for exact_names in exact_names_values:
        print(f"  \n Processing audio files: {exact_names}")
        df = humv_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)
        # df = df[df['Label'].isin([0, 2])]
        # df['Label'] = df['Label'].replace(2, 1)
        
        # List of patient IDs to include
        # patients_to_include_HC_vs_AC = ["HUMV_NFC_8","HUMV_HC_9","HUMV_NFC_21","HUMV_HC_17","HUMV_NFC_23","HUMV_NFC_22","HUMV_NFC_12","HUMV_NFC_6","HUMV_HC_22","HUMV_NFC_9","HUMV_NFC_23","HUMV_HC_19","HUMV_HC_11","HUMV_HC_16","HUMV_NFC_18","HUMV_NFC_24","HUMV_NFC_6","HUMV_HC_24","HUMV_HC_4","HUMV_HC_19","HUMV_AC_4","HUMV_AC_6","HUMV_AC_9","HUMV_AC_10","HUMV_AC_11","HUMV_AC_12","HUMV_AC_17","HUMV_AC_18","HUMV_AC_19","HUMV_AC_20","HUMV_AC_21","HUMV_AC_22","HUMV_AC_23","HUMV_AC_24","HUMV_AC_30","HUMV_AC_32", "HUMV_AC_34"]
        patients_to_include_AC_vs_PD = ["HUMV_PD_3","HUMV_PD_17","HUMV_PD_4","HUMV_PD_21","HUMV_PD_27","HUMV_PD_5","HUMV_PD_16","HUMV_PD_26","HUMV_PD_31","HUMV_PD_18","HUMV_PD_28","HUMV_PD_23","HUMV_PD_32","HUMV_PD_20","HUMV_PD_6","HUMV_PD_29","HUMV_PD_15","HUMV_AC_4","HUMV_AC_6","HUMV_AC_9","HUMV_AC_10","HUMV_AC_11","HUMV_AC_12","HUMV_AC_17","HUMV_AC_18","HUMV_AC_19","HUMV_AC_20","HUMV_AC_21","HUMV_AC_22","HUMV_AC_23","HUMV_AC_24","HUMV_AC_30","HUMV_AC_32","HUMV_AC_34"]

        print("Including patients:", patients_to_include_AC_vs_PD)
        
        # Remove duplicates just in case
        patients_to_include_AC_vs_PD = list(set(patients_to_include_AC_vs_PD))
        
        # Filter the dataframe
        df_filtered = df[df['Patient'].isin(patients_to_include_AC_vs_PD)].copy()

        value_counts = df_filtered['Label'].value_counts()
        print(value_counts) 

        audio_segments, labels_np, patient_ids, exercises = audio_processor.execute_preprocess_and_split(
            df_filtered,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=max_duration, # max_duration, None
            target_sr=16000,
            remove_silence=True,
            top_db=20,
            silence_duration=2,
            file_path_column='File_Path',
            label_column='Label',
            patient_column='Patient',
            audio_name_column='Audio_Name',
            overlap=0,
            min_chunk_ratio=0.75,
        )
        file_name_without_extension = os.path.splitext(exact_names)[0]
        np.save(os.path.join(output_folder, f"audio_segments_{output_extension}_{file_name_without_extension}.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_{output_extension}_{file_name_without_extension}.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_{output_extension}_{file_name_without_extension}.npy"), patient_ids)
        np.save(os.path.join(output_folder, f"exercises_{output_extension}_{file_name_without_extension}.npy"), exercises)
        print(f"Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")


Processing set: 1 audios of 5 seconds
  
 Processing audio files: aueoi.wav
Loaded 114 audio files.
Label distribution:
Label
0    49
1    34
2    31
Name: count, dtype: int64
Including patients: ['HUMV_PD_3', 'HUMV_PD_17', 'HUMV_PD_4', 'HUMV_PD_21', 'HUMV_PD_27', 'HUMV_PD_5', 'HUMV_PD_16', 'HUMV_PD_26', 'HUMV_PD_31', 'HUMV_PD_18', 'HUMV_PD_28', 'HUMV_PD_23', 'HUMV_PD_32', 'HUMV_PD_20', 'HUMV_PD_6', 'HUMV_PD_29', 'HUMV_PD_15', 'HUMV_AC_4', 'HUMV_AC_6', 'HUMV_AC_9', 'HUMV_AC_10', 'HUMV_AC_11', 'HUMV_AC_12', 'HUMV_AC_17', 'HUMV_AC_18', 'HUMV_AC_19', 'HUMV_AC_20', 'HUMV_AC_21', 'HUMV_AC_22', 'HUMV_AC_23', 'HUMV_AC_24', 'HUMV_AC_30', 'HUMV_AC_32', 'HUMV_AC_34']
Label
1    17
2    16
Name: count, dtype: int64
Total chunks generated: 61
Unique exercises: ['aueoi']
Arrays saved successfully in /home/marcos/Documentos/GitHub/TFM_code/data/processed/AC_vs_PD/5s_with_no_overlap_16kHz_top_db_20_mean_audio!
  
 Processing audio files: ka.wav
Loaded 118 audio files.
Label distribution:
Label
0    

# HC vs AC

In [8]:
import os
import sys

sys.path.insert(0, '/home/marcos/Documentos/GitHub/TFM_code/src')

from preprocessing import audio_processor
from data import humv_loader
import numpy as np
import pandas as pd

# Define root directory and output folder
root_directory = '/home/marcos/Documentos/GitHub/TFM_code/data/raw'

output_extension = '5s_with_no_overlap_16kHz_top_db_20_mean_audio'
output_folder = f'/home/marcos/Documentos/GitHub/TFM_code/data/processed/HC_vs_AC/{output_extension}'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
audio_length = 5  # seconds
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    (f"1 audios of {audio_length} seconds", ['aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav', 'ta.wav', 'uy.wav', 'vocal.wav'], 0, audio_length, 10),
    (f"3 audios of {audio_length} seconds", ['habla libre.wav', 'robo galletas.wav'], 0, audio_length, 35),
    (f"6 audios of {audio_length} seconds", ['fluencia categorial.wav'], 0, audio_length, 60),
    (f"10 audios of {audio_length} seconds", ['lectura texto.wav'], 0, audio_length, 150),
]

for desc, exact_names_values, start_time, chunk_duration, max_duration in exercise_sets:
    
    print(f"\nProcessing set: {desc}")
    
    for exact_names in exact_names_values:
        print(f"  \n Processing audio files: {exact_names}")
        df = humv_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)
        
        # List of patient IDs to include
        patients_to_include_HC_vs_AC = ["HUMV_NFC_8","HUMV_HC_9","HUMV_NFC_21","HUMV_HC_17","HUMV_NFC_23","HUMV_NFC_22","HUMV_NFC_12","HUMV_NFC_6","HUMV_HC_22","HUMV_NFC_9","HUMV_NFC_23","HUMV_HC_19","HUMV_HC_11","HUMV_HC_16","HUMV_NFC_18","HUMV_NFC_24","HUMV_NFC_6","HUMV_HC_24","HUMV_HC_4","HUMV_HC_19","HUMV_AC_4","HUMV_AC_6","HUMV_AC_9","HUMV_AC_10","HUMV_AC_11","HUMV_AC_12","HUMV_AC_17","HUMV_AC_18","HUMV_AC_19","HUMV_AC_20","HUMV_AC_21","HUMV_AC_22","HUMV_AC_23","HUMV_AC_24","HUMV_AC_30","HUMV_AC_32", "HUMV_AC_34"]
        print("Including patients:", patients_to_include_HC_vs_AC)
        
        # Remove duplicates just in case
        patients_to_include_HC_vs_AC = list(set(patients_to_include_HC_vs_AC))
        
        # Filter the dataframe
        df_filtered = df[df['Patient'].isin(patients_to_include_HC_vs_AC)].copy()

        value_counts = df_filtered['Label'].value_counts()
        print(value_counts) 

        audio_segments, labels_np, patient_ids, exercises = audio_processor.execute_preprocess_and_split(
            df_filtered,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=max_duration, # max_duration, None
            target_sr=16000,
            remove_silence=True,
            top_db=20,
            silence_duration=2,
            file_path_column='File_Path',
            label_column='Label',
            patient_column='Patient',
            audio_name_column='Audio_Name',
            overlap=0,
            min_chunk_ratio=0.75,
        )
        file_name_without_extension = os.path.splitext(exact_names)[0]
        np.save(os.path.join(output_folder, f"audio_segments_{output_extension}_{file_name_without_extension}.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_{output_extension}_{file_name_without_extension}.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_{output_extension}_{file_name_without_extension}.npy"), patient_ids)
        np.save(os.path.join(output_folder, f"exercises_{output_extension}_{file_name_without_extension}.npy"), exercises)
        print(f"Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")


Processing set: 1 audios of 5 seconds
  
 Processing audio files: aueoi.wav
Loaded 114 audio files.
Label distribution:
Label
0    49
1    34
2    31
Name: count, dtype: int64
Including patients: ['HUMV_NFC_8', 'HUMV_HC_9', 'HUMV_NFC_21', 'HUMV_HC_17', 'HUMV_NFC_23', 'HUMV_NFC_22', 'HUMV_NFC_12', 'HUMV_NFC_6', 'HUMV_HC_22', 'HUMV_NFC_9', 'HUMV_NFC_23', 'HUMV_HC_19', 'HUMV_HC_11', 'HUMV_HC_16', 'HUMV_NFC_18', 'HUMV_NFC_24', 'HUMV_NFC_6', 'HUMV_HC_24', 'HUMV_HC_4', 'HUMV_HC_19', 'HUMV_AC_4', 'HUMV_AC_6', 'HUMV_AC_9', 'HUMV_AC_10', 'HUMV_AC_11', 'HUMV_AC_12', 'HUMV_AC_17', 'HUMV_AC_18', 'HUMV_AC_19', 'HUMV_AC_20', 'HUMV_AC_21', 'HUMV_AC_22', 'HUMV_AC_23', 'HUMV_AC_24', 'HUMV_AC_30', 'HUMV_AC_32', 'HUMV_AC_34']
Label
0    17
1    17
Name: count, dtype: int64
Total chunks generated: 65
Unique exercises: ['aueoi']
Arrays saved successfully in /home/marcos/Documentos/GitHub/TFM_code/data/processed/HC_vs_AC/5s_with_no_overlap_16kHz_top_db_20_mean_audio!
  
 Processing audio files: ka.wav
Load